# EDA para Series de Tiempo
## ¿Tu serie es Estacionaria o tiene Tendencia?

> Análisis Exploratorio de Datos
> Dataset: Histórico de ventas sintético generado en el video anterior

---

### ¿Qué vamos a aprender?

Antes de aplicar cualquier modelo de forecasting, necesitamos responder una pregunta fundamental:

> **¿La serie de tiempo es estacionaria?**

Una serie es **estacionaria** cuando su **media**, **varianza** y **autocorrelación** son constantes en el tiempo.
La mayoría de los modelos clásicos (ARIMA, Holt-Winters) **lo requieren**. Si no lo verificamos, nuestros pronósticos no serán útiles.

### Ruta del notebook
1. Librerías y carga de datos
2. Exploración inicial de la serie
3. Prueba visual: Media y Desviación Estándar móviles
4. Pruebas estadísticas: ADF y KPSS
5. Tratamiento: Diferenciación para lograr estacionariedad
6. Autocorrelación (ACF): la firma visual de la estacionariedad
7. Conclusiones y próximos pasos

---
## 1. Librerías y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf
from scipy.stats import gaussian_kde
import warnings

# Ignorar advertencias de interpolación en KPSS para mantener la consola limpia
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.filterwarnings('ignore', category=InterpolationWarning)

# Estilo global de gráficas
plt.rcParams.update({
    'figure.facecolor': '#F8FAFC',
    'axes.facecolor':   '#F8FAFC',
    'axes.grid':        True,
    'grid.color':       '#E2E8F0',
    'grid.linewidth':   0.8,
    'font.size':        11,
})

BLUE   = '#2563EB'
RED    = '#DC2626'
GREEN  = '#16A34A'
PURPLE = '#7C3AED'

money_fmt = FuncFormatter(lambda x, _: f'${x/1e6:.1f}M')

print('✅ Librerías cargadas correctamente')

In [ ]:
from pathlib import Path

# Busca 'output/sales_history.csv'
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}' en output/. ")

csv_path = find_csv()
print(f'CSV encontrado en: {csv_path}')

df = pd.read_csv(csv_path, parse_dates=['date'])
df.sort_values('date', inplace=True)

print(f'Filas       : {len(df):,}')
print(f'Período     : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'SKUs únicos : {df["sku_id"].nunique()}')
print(f'Canales     : {df["channel"].unique().tolist()}')
df.head(3)

---
## 2. Construcción de la Serie de Tiempo

Para este análisis vamos a trabajar con los **ingresos semanales totales** — la suma de todos los SKUs y canales.
Esta es la serie que típicamente analizaría un analista antes de modelar la demanda agregada de la empresa.

In [ ]:
# Agregamos por semana: suma de ingresos de todos los SKUs y canales
# Nota: En pandas >=2.2 es recomendable usar .resample('W-SUN') para evitar warnings
serie = (
    df.groupby('date')['revenue']
    .sum()
    .resample('W-MON') 
    .sum()
)
serie.name = 'revenue'

print(f'Observaciones: {len(serie)}')
print(f'Media semanal: ${serie.mean():,.0f}')
print(f'Mín: ${serie.min():,.0f}')
print(f'Máx: ${serie.max():,.0f}')
serie.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(serie.index, serie.values, color=BLUE, linewidth=1.2, alpha=0.85)
ax.yaxis.set_major_formatter(money_fmt)
ax.set_title('Ingresos Semanales Totales — Serie Original', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Ingresos')
plt.tight_layout()
plt.show()

# ¿Que podemos observar? La serie sube año con año → Significa que hay tendencia.
# También hay picos repetidos (Buen Fin, Navidad) → Significa que hay estacionalidad.

---
## 3. Prueba Visual: Media y Desviación Estandar Móviles

La forma más intuitiva de detectar no-estacionariedad es graficar la **media móvil** y la **desviación estándar móvil**:

| Comportamiento | Interpretación |
|---|---|
| Media móvil **sube o baja** | ❌ La media no es constante → **tendencia** |
| Media móvil **plana** | ✅ Posiblemente estacionaria |
| Desv. estándar **crece** | ❌ Varianza no constante → **heterocedasticidad** |
| Desv. estándar **plana** | ✅ Varianza constante |

In [ ]:
WINDOW = 12  # 12 semanas ≈ 3 meses

roll_mean = serie.rolling(WINDOW).mean()
roll_std  = serie.rolling(WINDOW).std() 

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 1, hspace=0.45)

# Panel 1: Serie + media móvil
ax1 = fig.add_subplot(gs[0])
ax1.plot(serie.index, serie.values, color=BLUE, alpha=0.5, linewidth=1, label='Serie original')
ax1.plot(roll_mean.index, roll_mean.values, color=RED, linewidth=2.5,
         label=f'Media móvil ({WINDOW} semanas)')
ax1.yaxis.set_major_formatter(money_fmt)
ax1.set_title('¿La media es constante en el tiempo?', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left')

# Panel 2: Desv. estándar móvil
ax2 = fig.add_subplot(gs[1])
ax2.plot(roll_std.index, roll_std.values, color=GREEN, linewidth=2,
         label=f'Desv. estándar móvil ({WINDOW} semanas)')
ax2.yaxis.set_major_formatter(money_fmt)
ax2.set_title('¿La varianza es constante en el tiempo?', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left')

fig.suptitle('Prueba Visual de Estacionariedad', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Observa que la media móvil tiene pendiente positiva → tendencia creciente
# La desv. estándar también crece → varianza no constante

---
## 4. Pruebas Estadísticas: ADF y KPSS

La prueba visual es útil pero subjetiva. Las pruebas estadísticas nos dan un **veredicto objetivo**.

Usaremos **dos pruebas** con hipótesis opuestas — esto elimina la ambigüedad:

| Prueba | Hipótesis nula (H₀) | p-valor pequeño significa... |
|---|---|---|
| **ADF** | La serie **NO** es estacionaria | ✅ **SÍ** es estacionaria |
| **KPSS** | La serie **SÍ** es estacionaria | ❌ **NO** es estacionaria |

**Veredicto combinado:**

| ADF | KPSS | Conclusión |
|---|---|---|
| Rechaza H₀ | No rechaza H₀ | 🟢 Estacionaria |
| No rechaza H₀ | Rechaza H₀ | 🔴 No estacionaria |
| Ambas rechazan | — | 🟡 Ambiguo — revisar |
| Ninguna rechaza | — | 🟡 Ambiguo — revisar |

In [ ]:
def run_adf(s, alpha=0.05):
    stat, p, _, _, crit, _ = adfuller(s.dropna(), autolag='AIC')
    print('─' * 52)
    print('  PRUEBA ADF (Augmented Dickey-Fuller)')
    print('  H₀: La serie NO es estacionaria (tiene raíz unitaria)')
    print('─' * 52)
    print(f'  Estadístico : {stat:.4f}')
    print(f'  p-valor     : {p:.4f}')
    print(f'  Valores críticos: { {k: round(v,4) for k,v in crit.items()} }')
    if p < alpha:
        print(f'  ✅ p={p:.4f} < {alpha} → Se RECHAZA H₀ → Serie ESTACIONARIA')
    else:
        print(f'  ❌ p={p:.4f} ≥ {alpha} → NO se rechaza H₀ → Serie NO ESTACIONARIA')
    return p < alpha

adf_result = run_adf(serie)

In [ ]:
def run_kpss(s, alpha=0.05):
    stat, p, _, crit = kpss(s.dropna(), regression='ct', nlags='auto')
    print('─' * 52)
    print('  PRUEBA KPSS')
    print('  H₀: La serie SÍ es estacionaria')
    print('─' * 52)
    print(f'  Estadístico : {stat:.4f}')
    print(f'  p-valor     : {p:.4f}')
    print(f'  Valores críticos: { {k: round(v,4) for k,v in crit.items()} }')
    if p < alpha:
        print(f'  ❌ p={p:.4f} < {alpha} → Se RECHAZA H₀ → Serie NO ESTACIONARIA')
    else:
        print(f'  ✅ p={p:.4f} ≥ {alpha} → NO se rechaza H₀ → Serie ESTACIONARIA')
    return p >= alpha

kpss_result = run_kpss(serie)

In [ ]:
print('\n' + '═' * 52)
print('  VEREDICTO FINAL')
print('═' * 52)
if adf_result and kpss_result:
    print('  🟢 La serie ES ESTACIONARIA según ambas pruebas.')
elif not adf_result and not kpss_result:
    print('  🔴 La serie NO ES ESTACIONARIA.')
    print('     → Necesita diferenciación antes de modelar.')
else:
    print('  🟡 Resultado mixto. Analiza visualmente con más detalle.')
print('═' * 52)

---
## 5. Tratamiento: Diferenciación

La **diferenciación** es la técnica más común para eliminar tendencia y lograr estacionariedad.

$$\Delta y_t = y_t - y_{t-1}$$

En lugar de modelar el valor absoluto de ventas, modelamos el **cambio** entre períodos.  
Esto es exactamente el parámetro `d` en ARIMA(`p`, **`d`**, `q`).

In [ ]:
diff1 = serie.diff().dropna()
diff2 = serie.diff().diff().dropna()

fig, axes = plt.subplots(3, 1, figsize=(14, 11))

configs = [
    (serie, 'Serie Original', BLUE),
    (diff1, 'Primera Diferencia (d=1)', PURPLE),
    (diff2, 'Segunda Diferencia (d=2)', RED),
]

for ax, (s, title, color) in zip(axes, configs):
    ax.plot(s.index, s.values, color=color, linewidth=1.2)
    ax.axhline(s.mean(), color='black', linestyle='--',
               linewidth=1.2, alpha=0.7, label=f'Media: ${s.mean():,.0f}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.yaxis.set_major_formatter(money_fmt)
    ax.legend(loc='upper right', fontsize=9)

fig.suptitle('Efecto del Diferenciado sobre la Tendencia', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Verificamos que la primera diferencia ya es estacionaria
print('Pruebas sobre la PRIMERA DIFERENCIA:\n')
adf_d1   = run_adf(diff1)
print()
kpss_d1  = run_kpss(diff1)

---
## 6. ACF: La Firma Visual de la Estacionariedad

La **Función de Autocorrelación (ACF)** mide qué tan correlacionada está la serie consigo misma en distintos rezagos.

| Patrón en la ACF | Significado |
|---|---|
| Decaimiento **muy lento** | ❌ Serie no estacionaria — hay tendencia |
| Decaimiento **rápido** (primeros 2-3 lags) | ✅ Serie estacionaria |
| Picos cada **n períodos** | 🔄 Hay estacionalidad de período n |

In [ ]:
LAGS = 40

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(serie,  lags=LAGS, ax=ax1, color=BLUE,   alpha=0.05)
plot_acf(diff1,  lags=LAGS, ax=ax2, color=PURPLE,  alpha=0.05)

ax1.set_title('ACF — Serie Original\n→ Decaimiento lento = tendencia', 
              fontsize=12, fontweight='bold')
ax2.set_title('ACF — Primera Diferencia\n→ Decaimiento rápido = más estacionaria', 
              fontsize=12, fontweight='bold')

for ax in (ax1, ax2):
    ax.set_xlabel('Rezago (semanas)')
    ax.set_ylabel('Autocorrelación')

fig.suptitle('ACF: Antes y Después de Diferenciar', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#  En el próximo video veremos también la PACF para identificar los parámetros p y q de un modelo ARIMA.

---
## 7. Conclusiones

| Análisis | Resultado |
|---|---|
| Prueba visual (media/varianza móvil) |  Media creciente → tendencia presente |
| Prueba ADF (serie original) |  Estacionaria |
| Prueba KPSS (serie original) |  Estacionaria |
| Primera diferencia (Δ1) |  Estacionaria |
| ACF (serie original) |  Decaimiento lento |
| ACF (Δ1) |  Decaimiento rápido |

**¿Qué significa esto para el modelado?**

-  Si vamos a usar **ARIMA**, el parámetro `d=1`
-  Si vamos a usar **Prophet** o **XGBoost**, no necesitamos diferenciar — ellos manejan la tendencia internamente
-  La **estacionalidad** sigue presente incluso después de diferenciar — la trataremos en el **Video 3: Descomposición de Series**